# Row-Subset Single-Tree Contract for GBNet

This notebook is a readable, user-facing version of the contract test.

It does four things:
1. Shows the reference behavior we want from native XGBoost and LightGBM when a single tree is trained on only a chosen subset of rows.
2. Confirms that the existing GBNet full-data one-tree path still matches native full-data training.
3. Checks whether `gbnet` can reproduce the stricter subset-loss contract through `gb_step(row_indices=...)`.
4. Plots empirical Python timing for XGBoost subset updates as the total cached training size grows.

Today, the expected result is that the native reference sections work, XGBoost passes the stricter subset contract, LightGBM still reports missing support for subset updates, and the timing plot suggests that fixed-size XGBoost subset rounds stay much flatter than full-data rounds.

In [ ]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import perf_counter
import torch
import xgboost as xgb

from gbnet import lgbmodule as lgm
from gbnet import xgbmodule as xgm

np.set_printoptions(suppress=True, precision=6)
torch.set_printoptions(sci_mode=False, precision=6)

## Configurable fixture dataset

The notebook uses `n_rows = 1000` by default, but you can change that parameter in the next cell.

The selected training rows are every `subset_stride`-th row. Those rows follow a smooth signal. The held-out rows receive large alternating offsets. That keeps the contract informative at a more realistic scale: a tree trained on all rows should behave differently from a tree trained on only the selected rows.

In [ ]:
n_rows = 1000
subset_stride = 2
heldout_offset = 100.0


def make_row_subset_fixture(n_rows=1000, subset_stride=2, heldout_offset=100.0):
    X = np.arange(n_rows, dtype=np.float32).reshape(-1, 1)
    full_row_indices = np.arange(n_rows, dtype=np.int64)
    row_indices = np.arange(0, n_rows, subset_stride, dtype=np.int64)
    heldout_rows = np.setdiff1d(full_row_indices, row_indices)

    base_signal = (0.01 * X[:, 0] + 2.0 * np.sin(X[:, 0] / 40.0)).astype(np.float32)
    y = base_signal.copy()
    heldout_direction = np.where(((heldout_rows // subset_stride) % 2) == 0, 1.0, -1.0)
    y[heldout_rows] += (heldout_offset * heldout_direction).astype(np.float32)
    y = y.reshape(-1, 1)

    fixture = pd.DataFrame(
        {
            "row": full_row_indices,
            "feature": X[:, 0],
            "target": y[:, 0],
            "used_for_training": np.isin(full_row_indices, row_indices),
        }
    )
    return X, y, row_indices, heldout_rows, full_row_indices, fixture


X, y, row_indices, heldout_rows, full_row_indices, fixture = make_row_subset_fixture(
    n_rows=n_rows,
    subset_stride=subset_stride,
    heldout_offset=heldout_offset,
)

preview_count = min(8, n_rows)
preview_rows = np.unique(np.r_[0:preview_count, max(0, n_rows - preview_count) : n_rows])

print(
    f"n_rows={n_rows}, training_rows={len(row_indices)}, heldout_rows={len(heldout_rows)}, subset_stride={subset_stride}"
)
fixture.iloc[preview_rows]

## Native backend reference helpers

Each helper below trains exactly one tree on only the selected rows, then predicts for all rows.

That is the behavior GBNet's XGBoost row-subset update needs to reproduce, and the behavior a future LightGBM implementation would need to match.

In [ ]:
xgb_params = {
    "eta": 1.0,
    "max_depth": 2,
    "lambda": 0.0,
    "alpha": 0.0,
    "gamma": 0.0,
    "min_child_weight": 0.0,
    "tree_method": "hist",
    "nthread": 1,
}

lgb_params = {
    "learning_rate": 1.0,
    "num_leaves": 4,
    "max_depth": 2,
    "min_data_in_leaf": 1,
    "min_sum_hessian_in_leaf": 0.0,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "verbose": -1,
    "verbosity": -1,
    "num_threads": 1,
    "seed": 0,
}


def xgb_reference_predictions(X, y, row_indices, params):
    booster = xgb.train(
        {
            **params,
            "objective": "reg:squarederror",
            "base_score": 0.0,
        },
        xgb.DMatrix(X[row_indices], label=y[row_indices].reshape(-1)),
        num_boost_round=1,
    )
    return booster.predict(xgb.DMatrix(X)).reshape(-1, 1)


def lgb_reference_predictions(X, y, row_indices, params):
    booster = lgb.train(
        {
            **params,
            "objective": "regression",
            "boost_from_average": False,
        },
        lgb.Dataset(
            X[row_indices],
            label=y[row_indices].reshape(-1),
            init_score=np.zeros(len(row_indices), dtype=np.float32),
            free_raw_data=False,
        ),
        num_boost_round=1,
    )
    return booster.predict(X).reshape(-1, 1)

In [ ]:
comparison = fixture.copy()

comparison["xgb_subset_pred"] = xgb_reference_predictions(X, y, row_indices, xgb_params).flatten()
comparison["xgb_full_pred"] = xgb_reference_predictions(X, y, np.arange(X.shape[0]), xgb_params).flatten()
comparison["xgb_abs_diff"] = np.abs(comparison["xgb_subset_pred"] - comparison["xgb_full_pred"])

comparison["lgb_subset_pred"] = lgb_reference_predictions(X, y, row_indices, lgb_params).flatten()
comparison["lgb_full_pred"] = lgb_reference_predictions(X, y, np.arange(X.shape[0]), lgb_params).flatten()
comparison["lgb_abs_diff"] = np.abs(comparison["lgb_subset_pred"] - comparison["lgb_full_pred"])

comparison

## Sanity checks for the fixture

These numbers should be meaningfully above zero.

If they are, the fixture is informative: training on the subset is measurably different from training on all rows, and the subset-trained tree still produces predictions for held-out rows.

In [ ]:
reference_summary = pd.DataFrame(
    [
        {
            "backend": "xgboost",
            "max_abs_diff_subset_vs_full": float(comparison["xgb_abs_diff"].max()),
            "max_abs_pred_on_heldout_rows": float(np.abs(comparison.loc[heldout_rows, "xgb_subset_pred"]).max()),
        },
        {
            "backend": "lightgbm",
            "max_abs_diff_subset_vs_full": float(comparison["lgb_abs_diff"].max()),
            "max_abs_pred_on_heldout_rows": float(np.abs(comparison.loc[heldout_rows, "lgb_subset_pred"]).max()),
        },
    ]
)

reference_summary

## GBNet contract check

This cell uses the same setup as the test suite.

It checks two scenarios for each backend:
- `full_training_baseline`: train on all rows and verify that the existing GBNet path still matches native full-data one-tree training.
- `subset_loss_plus_row_indices`: train-time forward still uses the full `X`, but the loss only uses `preds[row_indices]` and `y[row_indices]`, then GBNet is asked to update with `gb_step(row_indices=row_indices)`.

Interpretation:
- `pass`: GBNet matched the native backend for that scenario.
- `missing_support`: this checkout of `gbnet` does not yet support the subset-loss contract.
- `mismatch`: GBNet ran, but its predictions did not match the native backend.

In [ ]:
def check_gbnet_full_contract(module_class, backend_name, params, expected_full_preds, loss_scale):
    module = module_class(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    module.train()
    preds = module(X)
    loss = loss_scale * loss_fn(preds, torch.tensor(y, dtype=torch.float))
    loss.backward(create_graph=True)
    module.gb_step()

    module.eval()
    actual_preds = module(X).detach().numpy()
    max_abs_error = float(np.max(np.abs(actual_preds - expected_full_preds)))
    status = "pass" if np.allclose(actual_preds, expected_full_preds, rtol=1e-6, atol=1e-6) else "mismatch"

    return {
        "backend": backend_name,
        "scenario": "full_training_baseline",
        "status": status,
        "max_abs_error": max_abs_error,
        "details": "" if status == "pass" else "Predictions did not match native full-data training.",
    }


def check_gbnet_subset_contract(module_class, backend_name, params, expected_subset_preds, loss_scale):
    module = module_class(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    try:
        module.train()
        preds = module(X)
        loss = loss_scale * loss_fn(preds[row_indices], torch.tensor(y[row_indices], dtype=torch.float))
        loss.backward(create_graph=True)
        module.gb_step(row_indices=row_indices)
    except Exception as exc:
        return {
            "backend": backend_name,
            "scenario": "subset_loss_plus_row_indices",
            "status": "missing_support",
            "max_abs_error": np.nan,
            "details": f"{type(exc).__name__}: {exc}",
        }

    module.eval()
    actual_preds = module(X).detach().numpy()
    max_abs_error = float(np.max(np.abs(actual_preds - expected_subset_preds)))
    status = "pass" if np.allclose(actual_preds, expected_subset_preds, rtol=1e-6, atol=1e-6) else "mismatch"

    return {
        "backend": backend_name,
        "scenario": "subset_loss_plus_row_indices",
        "status": status,
        "max_abs_error": max_abs_error,
        "details": "" if status == "pass" else "Predictions did not match native subset training.",
    }


gbnet_status = pd.DataFrame(
    [
        check_gbnet_full_contract(
            xgm.XGBModule,
            "xgboost",
            xgb_params,
            comparison[["xgb_full_pred"]].to_numpy(),
            loss_scale=0.5,
        ),
        check_gbnet_subset_contract(
            xgm.XGBModule,
            "xgboost",
            xgb_params,
            comparison[["xgb_subset_pred"]].to_numpy(),
            loss_scale=0.5,
        ),
        check_gbnet_full_contract(
            lgm.LGBModule,
            "lightgbm",
            lgb_params,
            comparison[["lgb_full_pred"]].to_numpy(),
            loss_scale=1.0,
        ),
        check_gbnet_subset_contract(
            lgm.LGBModule,
            "lightgbm",
            lgb_params,
            comparison[["lgb_subset_pred"]].to_numpy(),
            loss_scale=1.0,
        ),
    ]
)

gbnet_status

In [ ]:
full_statuses = set(gbnet_status.loc[gbnet_status["scenario"] == "full_training_baseline", "status"])
if full_statuses != {"pass"}:
    raise AssertionError("The existing full-data baseline no longer matches native one-tree training.")

xgb_subset_status = gbnet_status.loc[
    (gbnet_status["backend"] == "xgboost")
    & (gbnet_status["scenario"] == "subset_loss_plus_row_indices"),
    "status",
].iloc[0]
lgb_subset_status = gbnet_status.loc[
    (gbnet_status["backend"] == "lightgbm")
    & (gbnet_status["scenario"] == "subset_loss_plus_row_indices"),
    "status",
].iloc[0]

if xgb_subset_status != "pass":
    raise AssertionError("XGBoost row-subset training no longer matches native subset training.")
elif lgb_subset_status == "pass":
    print("GBNet row-subset single-tree training is working for both backends.")
elif lgb_subset_status == "missing_support":
    print("The existing full-data baseline is working.")
    print("XGBoost row-subset single-tree training is working.")
    print("LightGBM row-subset single-tree training is not implemented yet.")
else:
    raise AssertionError("LightGBM ran the subset contract, but it did not match native subset-trained predictions.")

## Empirical XGBoost timing

This section measures Python-visible wall time for one XGBoost `gb_step()` update.

It is not a proof of asymptotic complexity. The goal is simpler: show an empirical timing picture that is consistent with fixed-size subset updates staying much flatter than full-data rounds as the total cached training size grows.

The plot below keeps the subset size fixed and compares:
- `full_round`: `module.gb_step()` on all rows
- `subset_fixed_size`: `module.gb_step(row_indices=...)` on a fixed number of rows


In [ ]:
timing_n_rows_grid = [500, 1000, 2000, 4000, 8000]
timing_subset_size = 250
timing_repeats = 5


def _run_xgb_single_round(X, y, params, row_indices=None):
    module = xgm.XGBModule(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()

    module.train()
    preds = module(X)
    train_preds = preds if row_indices is None else preds[row_indices]
    train_y = y if row_indices is None else y[row_indices]
    loss = 0.5 * loss_fn(train_preds, torch.tensor(train_y, dtype=torch.float))
    loss.backward(create_graph=True)

    start = perf_counter()
    if row_indices is None:
        module.gb_step()
    else:
        module.gb_step(row_indices=row_indices)
    return perf_counter() - start

def measure_xgb_update_timing(n_rows_grid, subset_size, repeats, params):
    records = []
    for n_rows_i in n_rows_grid:
        X_i, y_i, _, _, _, _ = make_row_subset_fixture(
            n_rows=n_rows_i,
            subset_stride=subset_stride,
            heldout_offset=heldout_offset,
        )
        subset_rows_i = np.arange(0, subset_size * subset_stride, subset_stride, dtype=np.int64)
        if subset_rows_i[-1] >= n_rows_i:
            raise ValueError("subset_size is too large for one of the timing grid values")

        for _ in range(repeats):
            records.append(
                {
                    "n_rows": n_rows_i,
                    "scenario": "full_round",
                    "seconds": _run_xgb_single_round(X_i, y_i, params, row_indices=None),
                }
            )
            records.append(
                {
                    "n_rows": n_rows_i,
                    "scenario": f"subset_fixed_size_{subset_size}",
                    "seconds": _run_xgb_single_round(X_i, y_i, params, row_indices=subset_rows_i),
                }
            )
    return pd.DataFrame(records)


timing_results = measure_xgb_update_timing(
    n_rows_grid=timing_n_rows_grid,
    subset_size=timing_subset_size,
    repeats=timing_repeats,
    params=xgb_params,
)

timing_summary = timing_results.groupby(["n_rows", "scenario"], as_index=False).agg(
    mean_seconds=("seconds", "mean"),
    std_seconds=("seconds", "std"),
)

fig, ax = plt.subplots(figsize=(8, 5))
for scenario, label in [
    ("full_round", "full round (all rows)"),
    (f"subset_fixed_size_{timing_subset_size}", f"subset round ({timing_subset_size} rows)"),
]:
    curve = timing_summary[timing_summary["scenario"] == scenario].sort_values("n_rows")
    ax.plot(curve["n_rows"], curve["mean_seconds"] * 1000.0, marker="o", label=label)
    ax.fill_between(
        curve["n_rows"],
        (curve["mean_seconds"] - curve["std_seconds"].fillna(0.0)) * 1000.0,
        (curve["mean_seconds"] + curve["std_seconds"].fillna(0.0)) * 1000.0,
        alpha=0.15,
    )

ax.set_title("XGBoost gb_step timing: full rounds vs fixed-size subset rounds")
ax.set_xlabel("Total cached training rows")
ax.set_ylabel("Update wall time (ms)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

timing_summary

## Repeated rounds on one XGBoost model

This section keeps a single `XGBModule` alive across many boosting rounds and times the full training loop for each round.

Each round does exactly this, including `zero_grad()` at the start:

```python
module.train()
module.zero_grad()
preds = module(X)
loss = loss_scale * loss_fn(preds[row_indices], torch.tensor(y[row_indices], dtype=torch.float))
loss.backward(create_graph=True)
module.gb_step(row_indices=row_indices)
```

The comparison below runs the same loop with and without row subsetting and plots wall time by boosting round index.

Interpretation: if the lines stay relatively flat as the tree count grows, that is empirical evidence that repeated updates are not getting materially slower round by round.

In [ ]:
persistent_rounds = 30


def measure_repeated_xgb_rounds(X, y, params, loss_scale, n_rounds, row_indices=None):
    module = xgm.XGBModule(
        batch_size=X.shape[0],
        input_dim=X.shape[1],
        output_dim=1,
        params=params,
    )
    loss_fn = torch.nn.MSELoss()
    records = []

    for round_idx in range(1, n_rounds + 1):
        module.train()
        module.zero_grad()

        start = perf_counter()
        preds = module(X)
        train_preds = preds if row_indices is None else preds[row_indices]
        train_y = y if row_indices is None else y[row_indices]
        loss = loss_scale * loss_fn(train_preds, torch.tensor(train_y, dtype=torch.float))
        loss.backward(create_graph=True)
        if row_indices is None:
            module.gb_step()
            scenario = "full_rounds_same_model"
        else:
            module.gb_step(row_indices=row_indices)
            scenario = f"subset_rounds_same_model_{len(row_indices)}"
        elapsed = perf_counter() - start

        records.append(
            {
                "round": round_idx,
                "scenario": scenario,
                "seconds": elapsed,
            }
        )

    return pd.DataFrame(records)


repeated_round_results = pd.concat(
    [
        measure_repeated_xgb_rounds(
            X,
            y,
            xgb_params,
            loss_scale=0.5,
            n_rounds=persistent_rounds,
            row_indices=None,
        ),
        measure_repeated_xgb_rounds(
            X,
            y,
            xgb_params,
            loss_scale=0.5,
            n_rounds=persistent_rounds,
            row_indices=row_indices,
        ),
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(8, 5))
for scenario, label in [
    ("full_rounds_same_model", "full rounds on one model"),
    (f"subset_rounds_same_model_{len(row_indices)}", f"subset rounds on one model ({len(row_indices)} rows)"),
]:
    curve = repeated_round_results[repeated_round_results["scenario"] == scenario].sort_values("round")
    ax.plot(curve["round"], curve["seconds"] * 1000.0, marker="o", label=label)

ax.set_title("XGBoost repeated training rounds on one persistent model")
ax.set_xlabel("Boosting round")
ax.set_ylabel("Full round wall time (ms)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

repeated_round_results.pivot(index="round", columns="scenario", values="seconds") * 1000.0